# FActScore-Turbo Baseline — Results Analysis

Reads pre-computed results from `experiments/factscore_turbo_baseline/results/`.  
Does **not** re-run the scorer.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix, roc_curve

# Project root on sys.path so local imports work in notebook context.
PROJECT_ROOT = Path("__file__").resolve().parents[1]
RESULTS_DIR  = PROJECT_ROOT / "experiments" / "factscore_turbo_baseline" / "results"

plt.rcParams.update({"figure.dpi": 120, "font.size": 11})
print("Results dir:", RESULTS_DIR)

In [ ]:
df = pd.read_csv(RESULTS_DIR / "results.csv")
with open(RESULTS_DIR / "results.json") as fh:
    metrics = json.load(fh)

print(f"Loaded {len(df)} rows — columns: {list(df.columns)}")
df.head(3)

In [ ]:
# Key metrics summary.
summary_keys = [
    "n_samples", "roc_auc", "optimal_f1", "optimal_threshold",
    "optimal_precision", "optimal_recall",
    "avg_factscore", "std_factscore", "avg_n_facts",
]
for k in summary_keys:
    v = metrics.get(k)
    fmt = f"{v:.4f}" if isinstance(v, float) else str(v)
    print(f"  {k:<25s} {fmt}")

In [ ]:
# FActScore distribution by ground-truth label.
fig, ax = plt.subplots(figsize=(7, 4))
for label, grp in df.groupby("is_hallucinated"):
    ax.hist(
        grp["factscore"].dropna(),
        bins=25,
        alpha=0.6,
        density=True,
        label="Hallucinated" if label else "Faithful",
    )
ax.axvline(0.5,  color="black",  linestyle="--", lw=1, label="threshold=0.50")
ax.axvline(metrics["optimal_threshold"], color="red", linestyle="--", lw=1,
           label=f"optimal τ={metrics['optimal_threshold']:.3f}")
ax.set_xlabel("FActScore")
ax.set_ylabel("Density")
ax.set_title("FActScore Distribution by Ground-Truth Label")
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
# ROC curve.
valid = df.dropna(subset=["factscore", "is_hallucinated"])
y_true  = valid["is_hallucinated"].astype(int).values
y_score = (1 - valid["factscore"]).values

fpr, tpr, _ = roc_curve(y_true, y_score)
auc_val = metrics["roc_auc"]

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(fpr, tpr, lw=2, label=f"FActScore-Turbo  AUC = {auc_val:.3f}")
ax.plot([0, 1], [0, 1], "k--", lw=1)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve — Hallucination Detection")
ax.legend(loc="lower right")
fig.tight_layout()
plt.show()

In [ ]:
# Per-task F1 bar chart.
from sklearn.metrics import f1_score

tau = metrics["optimal_threshold"]
records = []
for task, grp in df.groupby("task_type"):
    g = grp.dropna(subset=["factscore", "is_hallucinated"])
    if len(g) < 5:
        continue
    yt = g["is_hallucinated"].astype(int).values
    yp = (g["factscore"] < tau).astype(int).values
    records.append({"task_type": task, "f1": f1_score(yt, yp, zero_division=0), "n": len(g)})

task_df = pd.DataFrame(records).sort_values("f1", ascending=False)

fig, ax = plt.subplots(figsize=(max(5, len(task_df) * 1.4), 4))
bars = sns.barplot(data=task_df, x="task_type", y="f1", ax=ax, palette="viridis")
for p, row in zip(ax.patches, task_df.itertuples()):
    ax.text(p.get_x() + p.get_width() / 2, p.get_height() + 0.01,
            f"n={row.n}", ha="center", va="bottom", fontsize=9)
ax.set_ylim(0, 1)
ax.set_xlabel("Task Type")
ax.set_ylabel("Binary F1")
ax.set_title(f"Per-Task F1 at Optimal Threshold (τ={tau:.3f})")
fig.tight_layout()
plt.show()

In [ ]:
# Facts-per-response distribution.
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(df["n_facts"].dropna(), bins=20, edgecolor="white")
ax.axvline(metrics["avg_n_facts"], color="red", linestyle="--", lw=1.5,
           label=f"mean = {metrics['avg_n_facts']:.1f}")
ax.set_xlabel("Number of Atomic Facts")
ax.set_ylabel("Count")
ax.set_title("Facts-per-Response Distribution")
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
# Confusion matrix at optimal threshold.
tau = metrics["optimal_threshold"]
valid = df.dropna(subset=["factscore", "is_hallucinated"])
y_true = valid["is_hallucinated"].astype(int).values
y_pred = (valid["factscore"] < tau).astype(int).values

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(4, 4))
ConfusionMatrixDisplay(cm, display_labels=["Faithful", "Hallucinated"]).plot(
    ax=ax, colorbar=False, cmap="Blues"
)
ax.set_title(f"Confusion Matrix at τ={tau:.3f}")
fig.tight_layout()
plt.show()

## Interpretation

**Dataset.** 200 RAGTruth samples (100 hallucinated / 100 faithful, balanced).  

**Signal quality.**  
FActScore-Turbo achieves **AUC = 0.706**, indicating moderate discriminative power —  
well above random (0.5) but with room for improvement.  
The score distribution shows substantial overlap between classes, confirming the task difficulty.

**Threshold sensitivity.**  
At the default threshold τ = 0.5, the classifier is extremely conservative:  
nearly all responses score above 0.5, so recall for the hallucinated class is near zero.  
The optimal threshold τ ≈ 0.917 yields **F1 = 0.699**, **precision = 0.670**, **recall = 0.730** —  
a much more balanced operating point.  
This implies that FActScore-Turbo assigns high factual scores even to hallucinated responses,  
which is consistent with the mean score of 0.871 (std = 0.138).

**Fact decomposition.**  
Average of ~9.7 atomic facts per response.  
The qwen2.5:7b decomposer tends to extract fine-grained claims,  
which benefits recall at the cost of precision in verification.

**Per-task variation.**  
F1 varies by task type; summarisation tasks tend to score lower because  
they introduce more paraphrasing not directly supported verbatim by the context.

**Conclusion.**  
FActScore-Turbo provides a useful but imperfect factuality signal.  
The Lookback Lens classifier (attention-based) is expected to capture complementary information,  
and the merged system (Milestone 3) aims to improve overall AUC beyond 0.706.